In [1]:
from langchain_openai import ChatOpenAI
from typing import Optional, Literal, List
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama.chat_models import ChatOllama
import requests
import pandas as pd
from tqdm import tqdm
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

class Grader(BaseModel):
    """Tool to asses wether an abstracts is related to neanderthals or not"""

    score : Literal["yes", "no"] = Field(description="the abstract talks about Neanderthals AND talks about an archaeological site AND carnivore remains or not. Use ‘yes’ if it does, use ‘no’ if it doesn't")
    topics: List[str] = Field(description="A list of topics the abstract discusses")

ollama = ChatOllama(
    model="llama3.2:1b-instruct-fp16",
    temperature=0,)

llm = ChatOpenAI(
    model="gpt-4o-mini"
    )

system_prompt = """
<role>
    Take on the role of an expert in analysing and understanding scientific articles and abstracts.
</role>

<taks>
    Please define if the following abstract talks about neanderthals AND talks about an archaeological site AND carnivore remains or not.
    It must talk about the three topics mentioned.
    In addition define the topics it discusses.
</task>

<abstract>
    {abstract}
</abstract>
"""

grader_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{abstract}")
    ]
)

structured_llm = llm.with_structured_output(Grader)
grader = grader_prompt | structured_llm

ollama_structured = ollama.with_structured_output(Grader)
ollama_router = grader_prompt | ollama_structured

In [2]:
# Define the API base URL
base_url = "https://api.litmaps.com/keywordSearch"

# Set up parameters
params = {
    "query": '"Pleistocene" "carnivores"',
    "useReranker": False,
    "showArticleDetails": True,
    "page": 1,          # Optional: set page number
    "per": 10000           # Optional: set number of results per page
}

# Perform the GET request
response = requests.get(base_url, params=params)

# Initialize an empty list to store article data
articles_data = []

# Check if the request was successful
if response.status_code == 200:
    data = response.json()
    # Extract relevant information
    for article in data['resultItems']:
        title = article.get('title', 'No Title')
        author_string = article.get('authorString', 'No Authors')
        publication_date = article.get('publicationDate', 'No Date')
        doi = article.get('doi', 'No DOI')
        url = article.get('url', 'No URL')
        forward_edge_count = article.get('forwardEdgeCount', 0)
        backward_edge_count = article.get('backwardEdgeCount', 0)
        publication_title = article.get('publicationTitle', 'No Publication Title')
        created_at = article.get('createdAt', 'No Date')
        updated_at = article.get('updatedAt', 'No Date')
        id = article.get("id", "no id")

        # Try to extract abstract, if not present, save the entire article object
        try:
            abstract = article['abstract']
            has_abstract = True
        except KeyError:
            abstract = str(article)  # Convert the entire article object to string
            has_abstract = False
        
        # Append the article data to the list
        articles_data.append({
            'title': title,
            'abstract': abstract,
            'has_abstract': has_abstract,
            'author_string': author_string,
            'publication_date': publication_date,
            'doi': doi,
            'url': url,
            'forward_edge_count': forward_edge_count,
            'backward_edge_count': backward_edge_count,
            'publication_title': publication_title,
            'created_at':created_at,
            'updated_at':updated_at,
            'id':id

        })

    # Create a DataFrame from the collected data
    articles_df_ = pd.DataFrame(articles_data)


else:
    print(f"Error: {response.status_code}")


Error: 504


In [ ]:
df = pd.concat([articles_df, articles_df_], axis=0, ignore_index=True)
df = df.drop_duplicates()
df.to_csv("carnivores-pleistocene.csv")

In [ ]:
### GRADE

# Initialize lists to store successful results and errors
results_data = []
errors_data = []

# Loop over the rows with abstracts
for index, row in tqdm(df[df["has_abstract"] == True].iterrows(), total=len(df[df["has_abstract"] == True])):
    try:
        # Attempt to invoke the grader
        results = grader.invoke({"abstract": row["abstract"]})

        # Append results to results_data if successful
        results_data.append({
            "index": index,
            "title": row["title"],
            "abstract": row["abstract"],
            "score": results.score,
            "topics": results.topics,
            "id": row.get("id", "No ID"),  # use get() in case column is missing
            "created_at": row.get("created_at", "No Created At"),
            "updated_at": row.get("updated_at", "No Updated At"),
            "author_string": row.get("author_string", "No Authors"),
            "publication_date": row.get("publication_date", "No Date")
        })

        # Print abstract if score is "yes"
        if results.score == "yes":
            print(row["abstract"])

    except Exception as e:
        print(e.args)
        errors_data.append({
            "index": index,
            "title": row["title"],
            "error_message": str(e),
            "abstract": row["abstract"]
        })
        continue

# Convert results to DataFrames
results_df = pd.DataFrame(results_data)
errors_df = pd.DataFrame(errors_data)

# Save DataFrames to CSV
results_df.to_csv("papers.csv", index=False)
results_df[results_df["score"] == "yes"].to_csv("papers.csv", index=False)
errors_df.to_csv("error_records.csv", index=False)



  5%|▍         | 349/7729 [05:31<2:29:44,  1.22s/it]

Les donnees taphonomiques et paleobiologiques quantifiees sur les tanieres d'ours des cavernes sont encore peu nombreuses. La caracterisation de ces tanieres necessite l'emploi de decomptes osseux exhaustifs et d'indices taphonomiques fiables. L'analyse de 30 472 restes osseux, pour un minimum de 487 ours des cavernes, repartis dans huit grottes du Pleistocene superieur du pourtour mediterraneen (Catalogne, Roussillon, Languedoc et Ligurie) permet de definir un ensemble de 24 indices pour distinguer les occupations ursines par rapport aux occupations carnivores et surtout humaines. Les particularites des accumulations osseuses creees par l'ours des cavernes tels que l'abondance des restes, la frequence des elements squelettiques et la structure des thanatocenoses sont examinees. Une approche paleocomportementale figuree par l'analyse des bioglyphes (bauges, griffades, empreintes) et des modifications osseuses provoquees par pietinement fournit une lecture nouvelle de la conservation, d

  7%|▋         | 536/7729 [08:27<2:00:23,  1.00s/it]

Nowadays, opportunistic small predators, such as foxes (Vulpes vulpes and Vulpes lagopus), are well known to be very adaptable to human modified ecosystems. However, the timing of the start of this phenomenon in terms of human impact on ecosystems and of the implications for foxes has hardly been studied. We hypothesize that foxes can be used as an indicator of past human impact on ecosystems, as a reflection of population densities and consequently to track back the influence of humans on the Pleistocene environment. To test this hypothesis, we used stable isotope analysis (δ13C, δ15N) of bone collagen extracted from faunal remains from several archaeological sites located in the Swabian Jura (southwest Germany) and covering a time range over three important cultural periods, namely the Middle Palaeolithic (older than 42,000 years ago) attributed to Neanderthals, and the early Upper Palaeolithic periods Aurignacian and Gravettian (42,000 to 30,000 years ago) attributed to modern human

 11%|█         | 833/7729 [12:56<1:50:05,  1.04it/s]

...

 87%|████████▋ | 6725/7729 [2:01:20<18:40,  1.12s/it]

Isotope and archeological analyses of Paleolithic food webs have suggested that Neandertal subsistence relied mainly on the consumption of large herbivores. This conclusion was primarily based on elevated nitrogen isotope ratios in Neandertal bone collagen and has been significantly debated. This discussion relies on the observation that similar high nitrogen isotopes values could also be the result of the consumption of mammoths, young animals, putrid meat, cooked food, freshwater fish, carnivores, or mushrooms. Recently, compound-specific C and N isotope analyses of bone collagen amino acids have been demonstrated to add significantly more information about trophic levels and aquatic food consumption. We undertook single amino acid C and N isotope analysis on two Neandertals, which were characterized by exceptionally high N isotope ratios in their bulk bone or tooth collagen. We report here both C and N isotope ratios on single amino acids of collagen samples for these two Neandertal

 99%|█████████▊| 7629/7729 [2:17:36<01:40,  1.00s/it]

Stable isotope analysis is increasingly used to gain insight in the configuration of Pleistocene ecosystems. The application of isotope analysis to Neanderthal and cave hyena bone assemblages has led to hypotheses about the niche differentiation between these species. Comparing isotopic data with archaeozoology analyses shows discrepancies between the results of both analytical methods. Here, the results of all northwest European stable isotope studies on Neanderthals are reviewed. The emphasis of the analysis is on a sample of sites from MIS 4-3 in southwest France. Causes of the discrepancy between archaeozoological and stable isotope results are discussed and hypotheses reconciling the data are proposed. Recommendations for further research will allow testing of the hypotheses and increase our understanding of the functioning of Pleistocene ecosystems.


100%|██████████| 7729/7729 [2:19:16<00:00,  1.08s/it]


In [84]:
# Specify the column names for the content and filenames
content_column = 'abstract'
filename_column = 'title'

# Loop through each row and save the content to a text file using the specified filename
for _, row in results_df[results_df["score"] == "yes"].iterrows():
    filename = f"{row[filename_column]}.txt"  # Use the filename from the specified column
    with open(filename, 'w') as file:
        file.write(row[content_column])